# Solutions · 12 · Queueing theory

Worked solutions to the exercises of [notebook 12](../notebooks/12_queueing_theory.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import queues as qu, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

A hospital ward has 30 beds, admissions are Poisson at 5 per day, stays average 5 days. Compute the probability that an admission is turned away (Erlang B) and the number of beds needed for 1 % blocking. Check one answer by simulating M/G/c/c with lognormal stays (insensitivity).

In [2]:
a = 5 * 5.0
print(f"30 beds, 25 erlangs: P(turned away) = {qu.erlang_b(30, a):.4f}")
c = next(c for c in range(1, 100) if qu.erlang_b(c, a) <= 0.01)
print(f"beds for at most 1 % blocking: {c} (Erlang B {qu.erlang_b(c, a):.4f})")
n = 200_000
arr = np.cumsum(R(1).exponential(1 / 5.0, n)); stay = R(2).lognormal(np.log(5.0) - 0.5 * 0.8**2, 0.8, n)
import heapq
beds = []; blocked = 0
for t, s in zip(arr, stay):
    while beds and beds[0] <= t: heapq.heappop(beds)
    if len(beds) < 30: heapq.heappush(beds, t + s)
    else: blocked += 1
print(f"simulated M/G/30/30 with lognormal stays (mean 5 days): blocking {blocked / n:.4f} - insensitive to the stay distribution")

30 beds, 25 erlangs: P(turned away) = 0.0526
beds for at most 1 % blocking: 36 (Erlang B 0.0080)
simulated M/G/30/30 with lognormal stays (mean 5 days): blocking 0.0542 - insensitive to the stay distribution


Erlang's loss formula depends on the service-time distribution only through its mean (Sevastyanov's
insensitivity theorem), so it applies to lognormal hospital stays: about 5 % of admissions are turned away with
30 beds, and 1 % blocking needs 36. The simulation keeps a heap of discharge times and confirms the formula.

## Exercise 2

Two call centres each receive 100 calls/h with 4-min calls and must answer 80 % in 20 s. Compare the agents needed separately and when pooled, and simulate the pooled centre with the measured handle times.

In [3]:
lam, mean_h = 100 / 60, 4.0
sep = qu.staffing(lam, 1 / mean_h, 0.8, 20 / 60)
pool = qu.staffing(2 * lam, 1 / mean_h, 0.8, 20 / 60)
print(f"separately: 2 x {sep['c']} = {2 * sep['c']} agents; pooled: {pool['c']} agents ({2 * sep['c'] - pool['c']} fewer)")
from engstoch import datasets
cc = pd.read_csv(datasets.path("call_centre.csv"), comment="#"); h = cc.handle_min[cc.handle_min > 0].to_numpy()
arr = np.cumsum(R(3).exponential(1 / (2 * lam), 300_000))
w = qu.simulate_ggc(arr, R(4).choice(h, 300_000), pool["c"])["wait"][20_000:]
print(f"pooled centre simulated with the measured handle times: P(wait <= 20 s) = {np.mean(w <= 20 / 60):.3f} (Erlang C predicts {pool['service_level']:.3f})")

separately: 2 x 10 = 20 agents; pooled: 17 agents (3 fewer)


pooled centre simulated with the measured handle times: P(wait <= 20 s) = 0.837 (Erlang C predicts 0.810)


Pooling saves agents because the variability of the combined load grows only like the square root of the
load. With the real (less variable than exponential) handle times the pooled centre does better than Erlang C
predicts - the exponential model is conservative here.

## Exercise 3

**Implement it yourself:** write the Lindley recursion and an estimate of P(W > t) for M/M/1, and compare with the exact rho e^{-(mu - lambda) t}.

In [4]:
lam, mu, n = 0.8, 1.0, 500_000
a = R(5).exponential(1 / lam, n); s = R(6).exponential(1 / mu, n)
w = np.zeros(n)
for k in range(1, n):
    w[k] = max(0.0, w[k - 1] + s[k - 1] - a[k])
w = w[20_000:]
for t in (0.0, 1.0, 5.0, 10.0):
    print(f"P(W > {t:4}) simulated {np.mean(w > t):.4f}, exact rho e^(-(mu - lambda) t) = {lam / mu * np.exp(-(mu - lam) * t):.4f}")

P(W >  0.0) simulated 0.7994, exact rho e^(-(mu - lambda) t) = 0.8000
P(W >  1.0) simulated 0.6541, exact rho e^(-(mu - lambda) t) = 0.6550
P(W >  5.0) simulated 0.2917, exact rho e^(-(mu - lambda) t) = 0.2943
P(W > 10.0) simulated 0.1070, exact rho e^(-(mu - lambda) t) = 0.1083


In M/M/1 the waiting time is 0 with probability 1 − ρ and otherwise exponential with rate μ − λ. The
recursion reproduces the tail, though slowly converging estimates at large t remind us that the waits are
strongly autocorrelated (notebook 14).